In [1]:
import numpy as np
import pandas as pd
# plotting imports and settings
import matplotlib
import matplotlib.pyplot as plt
from cycler import cycler


import tarfile
from pathlib import Path
import os
import json
from typing import List

from utils.metrics import experiment

plt.rcParams["xtick.direction"] = "in"
plt.clf()
plt.close('all')
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.major.size"] = 5.0
plt.rcParams["xtick.minor.size"] = 3.0
plt.rcParams["ytick.major.size"] = 5.0
plt.rcParams["ytick.minor.size"] = 3.0
plt.rc("font", family="serif", size=25)
plt.rcParams["ytick.minor.size"] = 3.0
matplotlib.rcParams.update(
    {"axes.grid": True, "grid.alpha": 0.75, "grid.linewidth": 0.5}
)

name = "tab20"
cmap = plt.get_cmap(name)  # type: matplotlib.colors.ListedColormap
colors_tab = cmap.colors  # type: list

matplotlib.rcParams["axes.prop_cycle"] = cycler(color=colors_tab,)

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]


In [ ]:
exp = experiment('/runs').load(with_transpiled_depth_ratio=True)

In [ ]:
print("Attributes of exp: \n", exp.__dict__.keys())
print("\nExp meta keys: \n", exp.meta.keys())

In [ ]:
anstzs = exp.meta.ansatz_name.unique()
targets = exp.meta.target_name.unique()
methods = exp.meta.method.unique()

index = pd.MultiIndex.from_product([targets, anstzs], names=['Target', 'Ansatz'])
columns = pd.MultiIndex.from_product([['HST', 'LET', 'QWC no op cycl', 'QWC op cycl'], ['Fixed', 'Sampling']], names=['method', 'fixed_states'])

def count_experiments(method, fixed, op_cycl=False):
    if method == 'wasserstein':
        return [exp.meta.loc[(exp.meta.target_name == t) & (exp.meta.ansatz_name == a) & (exp.meta.method == method) & (exp.meta.fixed_states == fixed) & (exp.meta.test_states == 'product') & (exp.meta.operator_cycling == op_cycl), 'C_final'].shape[0] 
            for t in targets for a in anstzs]
    else:
        return [exp.meta.loc[(exp.meta.target_name == t) & (exp.meta.ansatz_name == a) & (exp.meta.method == method) & (exp.meta.fixed_states == fixed) & (exp.meta.test_states == 'product'), 'C_final'].shape[0] 
            for t in targets for a in anstzs]

Num_HST_Exp_f1 = count_experiments('HST', True)
Num_HST_Exp_f2 = count_experiments('HST', False)
Num_LET_Exp_f1 = count_experiments('LET', True)
Num_LET_Exp_f2 = count_experiments('LET', False)
Num_QWC_Exp_f1 = count_experiments('wasserstein', True, False)
Num_QWC_Exp_f1_op_cycl = count_experiments('wasserstein', True, True)
Num_QWC_Exp_f2 = count_experiments('wasserstein', False, False)
Num_QWC_Exp_f2_op_cycl = count_experiments('wasserstein', False, True)

data = np.column_stack([Num_HST_Exp_f1, Num_HST_Exp_f2, Num_LET_Exp_f1, Num_LET_Exp_f2, Num_QWC_Exp_f1, Num_QWC_Exp_f2, Num_QWC_Exp_f1_op_cycl, Num_QWC_Exp_f2_op_cycl])
df = pd.DataFrame(data, index=index, columns=columns)
df.replace(0, np.nan, inplace=True)
new_df = df.dropna(how='all').fillna(0).astype(int)
new_df

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(10, 6), gridspec_kw={'wspace': 0.05, 'hspace': 0.05}, sharey=True)
target = 'HEA'
s = 8
n = 4
k = 2.0
fixed = True
entanglement = 'circular'
rep = '3'
condition_QWC = (exp.meta.n == n) & (exp.meta.target_name == target) & (exp.meta.ansatz_name == f'{target}-n{n}-rep{rep}-{entanglement}') & (exp.meta.s == s) & (exp.meta.fixed_states == fixed) & (exp.meta.operator_cycling == False)  & (exp.meta.test_states == 'product') & (exp.meta.method == 'wasserstein')

condition_LET = (exp.meta.n == n) & (exp.meta.target_name == target) & (exp.meta.ansatz_name == f'{target}-n{n}-rep{rep}-{entanglement}') & (exp.meta.s == s) & (exp.meta.fixed_states == fixed)  & (exp.meta.test_states == 'product') & (exp.meta.method == 'LET')

condition_HST = (exp.meta.n == n) & (exp.meta.target_name == target) & (exp.meta.ansatz_name == f'{target}-n{n}-rep{rep}-{entanglement}') & (exp.meta.s == s) & (exp.meta.fixed_states == fixed)  & (exp.meta.test_states == 'product') & (exp.meta.method == 'HST')

if fixed:
    fig.suptitle(f"Target: HEA-{entanglement}-rep{rep}, Qubits: {n}, Fixed States: {s}", bbox={'facecolor': 'white', 'alpha': 0.5, 'pad': 10}, position = (0.5, 1.05))
else:
    fig.suptitle(f"Target: HEA-{entanglement}-rep{rep}, Qubits: {n}, Sampling States: {s}")
    
QWC_sucess = np.count_nonzero(exp.meta.loc[condition_QWC, ['InFbar_final']].values < 1e-3)/len(exp.meta.loc[condition_QWC, ['InFbar_final']].values)

LET_succ = np.count_nonzero(exp.meta.loc[condition_LET, ['InFbar_final']].values < 1e-3)/len(exp.meta.loc[condition_LET, ['InFbar_final']].values)

HST_succ = np.count_nonzero(exp.meta.loc[condition_HST, ['InFbar_final']].values < 1e-3)/len(exp.meta.loc[condition_HST, ['InFbar_final']].values)

axs[0].loglog(1/exp.meta.loc[condition_QWC, ['C_final']], exp.meta.loc[condition_QWC, ['InFbar_final']], linestyle = 'None', marker = 'o', markersize = 15, alpha = 0.5, markeredgewidth = 2, markeredgecolor = 'k', markerfacecolor = 'blue')
axs[0].loglog(1/exp.C_EM.loc[:, condition_QWC], exp.InFbar.loc[:, condition_QWC], linestyle = 'None', marker = 'o', markerfacecolor = 'blue', markeredgecolor = 'blue', markersize = 2, alpha = 0.1)
axs[0].text(1e4, 1e-2, f"Success: {QWC_sucess*100:.1f}%", fontsize = 15, color = 'blue')
axs[0].set_ylabel(r"1-$\bar{F}$")
axs[0].set_title("QWC")

axs[1].loglog(1/exp.meta.loc[condition_LET, ['C_final']].values, exp.meta.loc[condition_LET, ['InFbar_final']].values, linestyle = 'None', marker = 'o', markerfacecolor = 'orange', markersize = 15, alpha = 0.5, markeredgewidth = 2, markeredgecolor = 'k')
axs[1].loglog(1/exp.C_LET.loc[:, condition_LET], exp.InFbar.loc[:, condition_LET], linestyle = 'None', marker = 'o', markerfacecolor = 'orange', markeredgecolor = 'orange', markersize = 2, alpha = 0.1)
axs[1].text(1e3, 1e-2, f"Success: {LET_succ*100:.1f}%", fontsize = 15, color = 'orange')
axs[1].set_title("LET")

axs[2].loglog(1/exp.meta.loc[condition_HST, ['C_final']].values, exp.meta.loc[condition_HST, ['InFbar_final']].values, linestyle = 'None', marker = 'o', markerfacecolor = 'green', markersize = 15, alpha = 0.5, markeredgewidth = 2, markeredgecolor = 'k')
axs[2].loglog(1/exp.C_HST.loc[:, condition_HST], exp.InFbar.loc[:, condition_HST], linestyle = 'None', marker = 'o', markerfacecolor = 'green', markeredgecolor = 'green', markersize = 2, alpha = 0.1)
axs[2].text(1e3, 1e-2, f"Success: {HST_succ*100:.1f}%", fontsize = 15, color = 'green')
axs[2].set_title("HST")

for i in range(3):
    axs[i].set_xlabel('Training Error$^{-1}$')